In [3]:
import pandas as pd
import numpy as np
import time
from tqdm import tqdm
import requests
import io

## Explanation

This file is taking the three csvs that contain time series data input at different cloudiness percentages (cloud_30_7.csv, cloud_50_8.csv, cloud_70_9.csv) and matching them with NASAs meteorological data to include: 

1. Wind speed at 2 Meters (2m above surface)
2. All Sky Surface Shortwave Downward Irradiance (solar exposure)
3. Precipitation Corrected (average total precipitation at the surface) --> sum of 3 days before

This will be loaded into csvs that can be used to train more accurate LSTM models

In [2]:
#script to matchup time series data & weather data from nasa
#from Izzy --> some slight adjustments to deal with change in time series data format 
def match_nasa(existing_df_path):
    """
    match nasa weather data with dates and locations in the existing dataset
    """

    # load final matched dataframe
    df = pd.read_csv(existing_df_path)
    df['satellite_date'] = pd.to_datetime(df['satellite_date'])
    df['caml_sample_date'] = pd.to_datetime(df['sample_date'])

    weather_results = []

    # iterate through unique location/date combos
    unique_matchups = df[['latitude', 'longitude', 'sample_date', 'satellite_date']].drop_duplicates()

    print(f"fetching weather for {len(unique_matchups)} matchups...")

    # for each unique matchup, get weather data
    for _, row in tqdm(unique_matchups.iterrows(), total=len(unique_matchups)):

        lat, lon = row['latitude'], row['longitude']

        # convert dates for nasa power
        date_str = row['satellite_date'].strftime('%Y%m%d')
        
        # create window 3 days before so we can calculate accumulated precipitation
        start_date = (row['satellite_date'] - pd.Timedelta(days=3)).strftime('%Y%m%d')
        
        url = f"https://power.larc.nasa.gov/api/temporal/daily/point"
        params = {
            "parameters": "PRECTOTCORR,WS2M,ALLSKY_SFC_SW_DWN",
            "community": "RE",
            "longitude": lon,
            "latitude": lat,
            "start": start_date,
            "end": date_str,
            "format": "JSON"
        }

        try:
            response = requests.get(url, params=params, timeout=10)
            if response.status_code == 200:

                # get data
                data = response.json()['properties']['parameter']
                
                # sum rain over the 3 days before
                precip_3d = sum(data['PRECTOTCORR'].values())

                # get wind and solar for the exact day
                wind = list(data['WS2M'].values())[-1]
                solar = list(data['ALLSKY_SFC_SW_DWN'].values())[-1]
                
                #add data to results
                weather_results.append({
                    'latitude': lat, 'longitude': lon, 'sample_date': row['sample_date'], 'satellite_date': row['satellite_date'],
                    'precip_3d_mm': precip_3d,
                    'wind_speed_ms': wind,
                    'solar_rad_wm2': solar
                })
            
            else:
                print(f"failed: Status {response.status_code} at {lat}, {lon}")

            time.sleep(0.2) 
            
        except Exception as e:
            print(lat, lon, row['satellite_date'])
            print("ERROR:", repr(e))
            continue

    # merge weather back into main dataframe
    weather_df = pd.DataFrame(weather_results)
    print(weather_df.columns)
    print(weather_df.shape)
    print(df.columns)
    print(df.shape)
    #merge after
    weather_df.to_csv("with_weather_50.csv", index=False)
    return weather_df

In [18]:
#modified version of match data to deal with previous weather matches 
#new match nasa function --> editing slightly
#script to matchup time series data & weather data from nasa
#from Izzy --> some slight adjustments to deal with change in time series data format 
def match_nasa2(df, name):
    """
    match nasa weather data with dates and locations in the existing dataset
    """
    
    # load final matched dataframe
    df['satellite_date'] = pd.to_datetime(df['satellite_date'])
    df['sample_date'] = pd.to_datetime(df['sample_date'])

    weather_results = []
    
    # iterate through unique location/date combos
    unique_matchups = df[['latitude', 'longitude', 'sample_date', 'satellite_date']].drop_duplicates()

    print(f"fetching weather for {len(unique_matchups)} matchups...")

    # for each unique matchup, get weather data
    for _, row in tqdm(unique_matchups.iterrows(), total=len(unique_matchups)):

        lat, lon = row['latitude'], row['longitude']

        # convert dates for nasa power
        date_str = row['satellite_date'].strftime('%Y%m%d')
        
        # create window 3 days before so we can calculate accumulated precipitation
        start_date = (row['satellite_date'] - pd.Timedelta(days=3)).strftime('%Y%m%d')
        
        url = f"https://power.larc.nasa.gov/api/temporal/daily/point"
        params = {
            "parameters": "PRECTOTCORR,WS2M,ALLSKY_SFC_SW_DWN",
            "community": "RE",
            "longitude": lon,
            "latitude": lat,
            "start": start_date,
            "end": date_str,
            "format": "JSON"
        }

        try:
            response = requests.get(url, params=params, timeout=10)
            if response.status_code == 200:

                # get data
                data = response.json()['properties']['parameter']
                
                # sum rain over the 3 days before
                precip_3d = sum(data['PRECTOTCORR'].values())

                # get wind and solar for the exact day
                wind = list(data['WS2M'].values())[-1]
                solar = list(data['ALLSKY_SFC_SW_DWN'].values())[-1]
                
                #add data to results
                weather_results.append({
                    'latitude': lat, 'longitude': lon, 'sample_date': row['sample_date'], 'satellite_date': row['satellite_date'],
                    'precip_3d_mm': precip_3d,
                    'wind_speed_ms': wind,
                    'solar_rad_wm2': solar
                })
            
            else:
                print(f"failed: Status {response.status_code} at {lat}, {lon}")

            time.sleep(0.2) 
            
        except Exception as e:
            print(lat, lon, row['satellite_date'])
            print("ERROR:", repr(e))
            continue

    # merge weather back into main dataframe
    weather_df = pd.DataFrame(weather_results)
    print(weather_df.columns)
    print(weather_df.shape)
    print(df.columns)
    print(df.shape)
    #merge after
    weather_df.to_csv(name, index=False)
    return weather_df

In [30]:
cloud_30 = pd.read_csv('cloud_30_7.csv')
print(cloud_30.shape)

(33509, 20)


In [5]:
#get matchup with 30 percent --> already ran this in a different file (including here for consistency)
#df_match = match_nasa('/Users/emersonfrost/Desktop/MLfCC/mlcc-final-project/before_30_7.csv')
#results from that match:
df_match = pd.read_csv('/Users/emersonfrost/Desktop/MLfCC/mlcc-final-project/with_weather_30.csv')
print(df_match.shape)
#mapping 

(30499, 7)


In [20]:
#draw from masterlist of matches (most recent call) + this + 50
#previous call to match with 50
df_match2 = pd.read_csv('/Users/emersonfrost/Desktop/MLfCC/mlcc-final-project/with_weather_50.csv')
print(df_match2.shape)
#previous call for forecast 1 week model
df_match3 = pd.read_csv('/Users/emersonfrost/Desktop/MLfCC/mlcc-final-project/week1_weather_restofmatches.csv')
print(df_match3.shape)

(41040, 7)
(13198, 7)


In [21]:
#combine all of these to create 'master' of matches
match_master = pd.concat([df_match, df_match2], ignore_index = True)
match_master = pd.concat([match_master, df_match3], ignore_index = True)
match_master = match_master.drop_duplicates()

In [22]:
print(match_master.shape) #have a total of 56279 weather matches as of now

(56279, 7)


In [23]:
def find_matches(df, df_match):
    join_cols = ['latitude', 'longitude', 'sample_date', 'satellite_date']
    
    #in datetime
    df['sample_date'] = pd.to_datetime(df['sample_date'])
    df_match['sample_date'] = pd.to_datetime(df_match['sample_date'])
    df['satellite_date'] = pd.to_datetime(df['satellite_date'])
    df_match['satellite_date'] = pd.to_datetime(df_match['satellite_date'])

    #rows with a match
    df_ww = df.merge(
        df_match,
        on=join_cols,
        how='inner'
    )
    #rows without a match
    df_nw = df.merge(
        df_match[join_cols],
        on=join_cols,
        how='left',
        indicator=True
    ).query('_merge == "left_only"').drop(columns=['_merge'])

    return df_ww, df_nw

In [24]:
#update master with most recent call
new_match = pd.read_csv('/Users/emersonfrost/Desktop/MLfCC/mlcc-final-project/week2_weather_restofmatches.csv')
print(new_match.shape)
print(new_match.isna().sum().sum())

(3338, 7)
0


In [26]:
match_master = pd.concat([match_master, new_match], ignore_index = True).drop_duplicates()
print(match_master.shape) #59242 total matches

(59242, 7)


In [31]:
#use this and match_master to see what we are missing for the cloud30
cloud_30_ww, cloud_30_nw = find_matches(cloud_30, match_master)
print(f"Shape of Matched Df: {cloud_30_ww.shape}")
print(f"Missing rows: {np.abs(cloud_30_ww.shape[0] - cloud_30.shape[0])}")
print(f"Shape of Non-Matched Df: {cloud_30_nw.shape}")
cloud_30_ww.head(5)

Shape of Matched Df: (32869, 23)
Missing rows: 640
Shape of Non-Matched Df: (640, 20)


,system:index,NIR,SWIR,blue,cloud_score,cyanobacteria_abundance,green,latitude,longitude,ndci,...,red,red_edge,sample_date,satellite_date,.geo,mean_diff_days,std_diff_days,precip_3d_mm,wind_speed_ms,solar_rad_wm2
0,8665_20211215T185811_20211215T190030_T10SFF,0.102973,0.116336,0.041726,0.084986,5427.50,0.062534,36.772302,-121.788002,0.382538,...,0.076428,0.162559,2021-12-29,2021-12-15,"{""type"":""MultiPoint"",""coordinates"":[]}",5.833333,2.041241,94.83,1.97,2.6995
1,8664_20211215T185811_20211215T190030_T10SFF,0.110361,0.147759,0.038600,0.084986,2417.50,0.063480,36.796199,-121.781996,0.518339,...,0.062576,0.190656,2021-12-29,2021-12-15,"{""type"":""MultiPoint"",""coordinates"":[]}",5.833333,2.041241,94.83,1.97,2.6995
2,8388_20211212T184801_20211212T185004_T11SKU,0.126156,0.228897,0.058332,0.000025,3025.25,0.082291,34.416800,-119.544000,0.323291,...,0.104801,0.202037,2021-12-16,2021-12-12,"{""type"":""MultiPoint"",""coordinates"":[]}",6.666667,2.581989,3.17,0.79,3.3838
3,8393_20211212T184801_20211212T185004_T10SGD,0.205398,0.347231,0.103124,0.000034,3162.50,0.144840,34.977198,-120.569999,0.195543,...,0.172852,0.255201,2021-12-14,2021-12-12,"{""type"":""MultiPoint"",""coordinates"":[]}",6.666667,4.082483,2.69,2.09,3.2962
4,8398_20211212T184801_20211212T185004_T10SGD,0.077875,0.138985,0.020431,0.000034,3141.00,0.039706,35.124500,-120.569000,0.704522,...,0.037623,0.210367,2021-12-14,2021-12-12,"{""type"":""MultiPoint"",""coordinates"":[]}",6.666667,4.082483,2.69,2.09,2.6167


In [29]:
#making the call to get the last 640 matches to complete 30% cloudiness file then create csv
#print(cloud_30_nw.shape)
#cloud_30_weather = match_nasa2(cloud_30_nw, "restof_30_matches.csv")

(640, 22)
fetching weather for 640 matchups...


100%|█████████████████████████████████████████| 640/640 [09:20<00:00,  1.14it/s]

Index(['latitude', 'longitude', 'sample_date', 'satellite_date',
       'precip_3d_mm', 'wind_speed_ms', 'solar_rad_wm2'],
      dtype='object')
(640, 7)
Index(['system:index', 'NIR', 'SWIR', 'blue', 'cloud_score',
       'cyanobacteria_abundance', 'green', 'latitude', 'longitude', 'ndci',
       'ndti', 'ndvi', 'ndwi', 'red', 'red_edge', 'sample_date',
       'satellite_date', '.geo', 'mean_diff_days_x', 'std_diff_days_x',
       'mean_diff_days_y', 'std_diff_days_y'],
      dtype='object')
(640, 22)


In [33]:
#now merge cloud_30_weather with cloud_30_nw
print(cloud_30_nw.shape)
join_cols = ['latitude', 'longitude', 'sample_date', 'satellite_date']
cloud_30_pt2 = cloud_30_nw.merge(cloud_30_weather, on = join_cols, how = 'left')
print(cloud_30_pt2.shape)
print(cloud_30_pt2.isna().sum().sum())

(640, 20)
(640, 23)
0


In [34]:
#concat with ww
cloud_30_pt2 = cloud_30_pt2[cloud_30_ww.columns]
cloud_30_7_ww = pd.concat([cloud_30_ww, cloud_30_pt2], ignore_index = True)
print(cloud_30_7_ww.shape)
print(cloud_30_7_ww.isna().sum().sum())

(33509, 23)
0


In [37]:
#return csv
cloud_30_7_ww = cloud_30_7_ww.sort_values(by = ['latitude', 'longitude', 'sample_date', 'cyanobacteria_abundance', 'satellite_date'])
print(cloud_30_7_ww.shape)

(33509, 23)


In [38]:
cloud_30_7_ww.to_csv('cloud_30_7_ww.csv', index = False)

#### 50% dataset --> map to weather

In [39]:
cloud_50 = pd.read_csv('cloud_50_8.csv')
print(cloud_50.shape)

(43184, 20)


In [40]:
#update matches
match_master = pd.concat([match_master, cloud_30_weather], ignore_index = True).drop_duplicates()
print(match_master.shape)

(59882, 7)


In [41]:
#use this and match_master to see what we are missing for the cloud30
cloud_50_ww, cloud_50_nw = find_matches(cloud_50, match_master)
print(f"Shape of Matched Df: {cloud_50_ww.shape}")
print(f"Missing rows: {np.abs(cloud_50_ww.shape[0] - cloud_50.shape[0])}")
print(f"Shape of Non-Matched Df: {cloud_50_nw.shape}")
cloud_50_ww.head(5)

Shape of Matched Df: (43019, 23)
Missing rows: 165
Shape of Non-Matched Df: (165, 20)


,system:index,NIR,SWIR,blue,cloud_score,cyanobacteria_abundance,green,latitude,longitude,ndci,...,red,red_edge,sample_date,satellite_date,.geo,mean_diff_days,std_diff_days,precip_3d_mm,wind_speed_ms,solar_rad_wm2
0,8664_20211215T185811_20211215T190030_T10SFF,0.110361,0.147759,0.038600,0.084986,2417.5,0.063480,36.796199,-121.781996,0.518339,...,0.062576,0.190656,2021-12-29,2021-12-15,"{""type"":""MultiPoint"",""coordinates"":[]}",5.714286,1.889822,94.83,1.97,2.6995
1,8665_20211215T185811_20211215T190030_T10SFF,0.102973,0.116336,0.041726,0.084986,5427.5,0.062534,36.772302,-121.788002,0.382538,...,0.076428,0.162559,2021-12-29,2021-12-15,"{""type"":""MultiPoint"",""coordinates"":[]}",5.714286,1.889822,94.83,1.97,2.6995
2,8394_20211212T184801_20211212T185004_T10SGD,0.110934,0.228372,0.038458,0.000034,3792.0,0.065363,34.957600,-120.630995,0.601167,...,0.065528,0.271334,2021-12-17,2021-12-12,"{""type"":""MultiPoint"",""coordinates"":[]}",5.714286,1.889822,2.69,2.09,3.2962
3,8386_20211212T184801_20211212T185004_T10SGD,0.142031,0.301194,0.056606,0.000034,3599.0,0.088971,35.016398,-120.585998,0.161824,...,0.126058,0.174855,2021-12-15,2021-12-12,"{""type"":""MultiPoint"",""coordinates"":[]}",5.714286,1.889822,2.69,2.09,2.6167
4,8395_20211212T184801_20211212T185004_T10SGD,0.059302,0.119080,0.006354,0.000034,3143.0,0.023984,35.022801,-120.585998,0.832178,...,0.019748,0.212595,2021-12-15,2021-12-12,"{""type"":""MultiPoint"",""coordinates"":[]}",5.714286,1.889822,2.69,2.09,2.6167


In [42]:
#now make call to get matches for 50%
print(cloud_50_nw.shape)
cloud_50_weather = match_nasa2(cloud_50_nw, "restof_50_matches.csv")

(165, 20)
fetching weather for 165 matchups...


100%|█████████████████████████████████████████| 165/165 [02:19<00:00,  1.19it/s]

Index(['latitude', 'longitude', 'sample_date', 'satellite_date',
       'precip_3d_mm', 'wind_speed_ms', 'solar_rad_wm2'],
      dtype='object')
(165, 7)
Index(['system:index', 'NIR', 'SWIR', 'blue', 'cloud_score',
       'cyanobacteria_abundance', 'green', 'latitude', 'longitude', 'ndci',
       'ndti', 'ndvi', 'ndwi', 'red', 'red_edge', 'sample_date',
       'satellite_date', '.geo', 'mean_diff_days', 'std_diff_days'],
      dtype='object')
(165, 20)


In [47]:
#now merge cloud_50_weather with cloud_50_nw
print(cloud_50_nw.shape)
join_cols = ['latitude', 'longitude', 'sample_date', 'satellite_date']
cloud_50_pt2 = cloud_50_nw.merge(cloud_50_weather, on = join_cols, how = 'left')
print(cloud_50_pt2.shape)
print(cloud_50_pt2.isna().sum().sum())

(165, 20)
(165, 23)
0


In [48]:
#concat with ww
cloud_50_pt2 = cloud_50_pt2[cloud_50_ww.columns]
cloud_50_8_ww = pd.concat([cloud_50_ww, cloud_50_pt2], ignore_index = True)
print(cloud_50_8_ww.shape)
print(cloud_50_8_ww.isna().sum().sum())

(43184, 23)
0


In [49]:
#return csv
cloud_50_8_ww = cloud_50_8_ww.sort_values(by = ['latitude', 'longitude', 'sample_date', 'cyanobacteria_abundance', 'satellite_date'])
cloud_50_8_ww.to_csv('cloud_50_8_ww.csv', index = False)

#### 70% dataset --> map to weather

In [50]:
cloud_70 = pd.read_csv('cloud_70_9.csv')
print(cloud_70.shape)

(50580, 20)


In [51]:
#update matches
match_master = pd.concat([match_master, cloud_50_weather], ignore_index = True).drop_duplicates()
print(match_master.shape)

(60047, 7)


In [52]:
#use this and match_master to see what we are missing for the cloud70
cloud_70_ww, cloud_70_nw = find_matches(cloud_70, match_master)
print(f"Shape of Matched Df: {cloud_70_ww.shape}")
print(f"Missing rows: {np.abs(cloud_70_ww.shape[0] - cloud_70.shape[0])}")
print(f"Shape of Non-Matched Df: {cloud_70_nw.shape}")
cloud_70_ww.head(5)

Shape of Matched Df: (49368, 23)
Missing rows: 1212
Shape of Non-Matched Df: (1212, 20)


,system:index,NIR,SWIR,blue,cloud_score,cyanobacteria_abundance,green,latitude,longitude,ndci,...,red,red_edge,sample_date,satellite_date,.geo,mean_diff_days,std_diff_days,precip_3d_mm,wind_speed_ms,solar_rad_wm2
0,8665_20211215T185811_20211215T190030_T10SFF,0.102973,0.116336,0.041726,0.084986,5427.5,0.062534,36.772302,-121.788002,0.382538,...,0.076428,0.162559,2021-12-29,2021-12-15,"{""type"":""MultiPoint"",""coordinates"":[]}",5.625,1.767767,94.83,1.97,2.6995
1,8664_20211215T185811_20211215T190030_T10SFF,0.110361,0.147759,0.038600,0.084986,2417.5,0.063480,36.796199,-121.781996,0.518339,...,0.062576,0.190656,2021-12-29,2021-12-15,"{""type"":""MultiPoint"",""coordinates"":[]}",5.625,1.767767,94.83,1.97,2.6995
2,8399_20211212T184801_20211212T185004_T10SGD,0.158283,0.266503,0.068509,0.000034,4174.5,0.101243,34.945400,-120.416998,0.323920,...,0.125662,0.241558,2021-12-17,2021-12-12,"{""type"":""MultiPoint"",""coordinates"":[]}",5.625,1.767767,2.69,2.09,3.2962
3,8395_20211212T184801_20211212T185004_T10SGD,0.059302,0.119080,0.006354,0.000034,3143.0,0.023984,35.022801,-120.585998,0.832178,...,0.019748,0.212595,2021-12-15,2021-12-12,"{""type"":""MultiPoint"",""coordinates"":[]}",5.625,1.767767,2.69,2.09,2.6167
4,8396_20211212T184801_20211212T185004_T11SKU,0.057452,0.104528,0.013154,0.000025,4890.0,0.027091,34.438400,-119.905999,0.775846,...,0.025192,0.187166,2021-12-16,2021-12-12,"{""type"":""MultiPoint"",""coordinates"":[]}",5.000,0.000000,2.45,1.78,3.3838


In [53]:
#matching for 70
print(cloud_70_nw.shape)
cloud_70_weather = match_nasa2(cloud_70_nw, "restof_70_matches.csv")

(1212, 20)
fetching weather for 1212 matchups...


100%|███████████████████████████████████████| 1212/1212 [17:20<00:00,  1.16it/s]

Index(['latitude', 'longitude', 'sample_date', 'satellite_date',
       'precip_3d_mm', 'wind_speed_ms', 'solar_rad_wm2'],
      dtype='object')
(1212, 7)
Index(['system:index', 'NIR', 'SWIR', 'blue', 'cloud_score',
       'cyanobacteria_abundance', 'green', 'latitude', 'longitude', 'ndci',
       'ndti', 'ndvi', 'ndwi', 'red', 'red_edge', 'sample_date',
       'satellite_date', '.geo', 'mean_diff_days', 'std_diff_days'],
      dtype='object')
(1212, 20)


In [54]:
#now merge cloud_70_weather with cloud_70_nw
print(cloud_70_nw.shape)
join_cols = ['latitude', 'longitude', 'sample_date', 'satellite_date']
cloud_70_pt2 = cloud_70_nw.merge(cloud_70_weather, on = join_cols, how = 'left')
print(cloud_70_pt2.shape)
print(cloud_70_pt2.isna().sum().sum())

(1212, 20)
(1212, 23)
0


In [55]:
#concat with ww
cloud_70_pt2 = cloud_70_pt2[cloud_70_ww.columns]
cloud_70_9_ww = pd.concat([cloud_70_ww, cloud_70_pt2], ignore_index = True)
print(cloud_70_9_ww.shape)
print(cloud_70_9_ww.isna().sum().sum())

(50580, 23)
0


In [56]:
#return csv
cloud_70_9_ww = cloud_70_9_ww.sort_values(by = ['latitude', 'longitude', 'sample_date', 'cyanobacteria_abundance', 'satellite_date'])
cloud_70_9_ww.to_csv('cloud_70_9_ww.csv', index = False)

In [57]:
#produce a csv that has final matches
final_matches = pd.concat([match_master, cloud_70_weather], ignore_index = True).drop_duplicates()
print(final_matches.shape)

(61259, 7)


In [58]:
#return this as a csv to call when making weather matches later
final_matches.to_csv('all_matches_from_ts.csv', index = False)